# 230 — Blob clustering

Valley-blob segmentation + weighted blob-feature clustering (KMeans K-sweep + Hierarchical K-sweep).
Operates on the **canonical dataset** (built once by `lf_dataset.prepare_dataset` — same samples as 210/231/232).
Outputs land in `outputs/clustering/{kmeans,hierarchical}/blob/runs/<timestamp>/`.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions.lf_blob_metrics import (
    s22_build_blob_feature_matrix,
    s23_init_blob_metric,
    render_blob_overlay,
    save_sample_blob_png,
    BLOB_WEIGHT_VEC,
)
from functions.lf_minus101 import save_sample_minus101_png
from functions import lf_cluster_run as R

SCRIPT_NAME = '230_blob_clustering.ipynb'


## Config

In [ ]:
# ── data input ───────────────────────────────
INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── valley blob segmentation params ─────────
VALLEY_PARAMS = cfg.VALLEY_PARAMS

# ── feature weighting ────────────────────────
FEATURE_TYPE_WEIGHTS = cfg.FEATURE_TYPE_WEIGHTS

# ── clustering ───────────────────────────────
KMEANS_K_RANGE = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
HC_METHOD      = 'average'
HC_METRIC      = 'euclidean'
RANDOM_STATE   = cfg.RANDOM_STATE

print('VALLEY_PARAMS:', VALLEY_PARAMS)
print('KMEANS_K_RANGE:', KMEANS_K_RANGE)


## Load canonical dataset

Shared across 210/230/231/232 so cross-feature comparisons are valid.


In [ ]:
# Loads ERSPs from INPUT_DIR, drops non-neural channels, applies the
# high-activity gate. Single source of truth so all four clustering
# notebooks operate on the same sample set. Cached in
# 02_FBM_Clustering/outputs/_dataset/canonical/ for fast reload.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')
df_meta.head()


## Valley blob segmentation + feature extraction

Builds the blob feature matrix `X_blob` and the per-sample blob list `blobs_per_sample`.
All canonical samples enter — no further gating (samples without strong blobs land in
low-activity blob clusters naturally).


In [ ]:
X_blob, max_scores, blobs_per_sample = s22_build_blob_feature_matrix(
    ersp_list=ersp_list,
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    thr_pos=float(VALLEY_PARAMS['thr_pos']),
    thr_neg=float(VALLEY_PARAMS['thr_neg']),
    delta_valley=float(VALLEY_PARAMS['delta_valley']),
    min_mean_pos=float(VALLEY_PARAMS['min_mean_pos']),
    max_mean_neg=float(VALLEY_PARAMS['max_mean_neg']),
    sign_mode=str(VALLEY_PARAMS['sign_mode']),
)
print('X_blob:', X_blob.shape, '  max_scores range:', max_scores.min(), max_scores.max())


## Per-sample alternate views (BLOB + minus101 PNGs)

Writes alternate visualizations alongside ERSP_clean for the MOBA samples-pane toggle:
* `ERSP_blob/<cond>/<stem>_BLOB.png` — transparent ERSP + q34-style blob overlays
* `ERSP_minus101/<cond>/<stem>_M101.png` — painted -1/0/+1 segmentation

Idempotent (skip-if-exists by default). Set `FORCE_REGEN=True` to overwrite.


In [ ]:
GEN_ALT_VIEWS = True
FORCE_REGEN   = False

def _alt_view_paths(file_path):
    """ERSP_matrix/<cond>/<stem>.npy -> the BLOB and M101 PNG paths."""
    p = str(file_path).replace('\\\\', '/').replace('\\', '/')
    if '/ERSP_matrix/' not in p:
        return None, None
    blob_p = p.replace('/ERSP_matrix/', '/ERSP_blob/').replace('.npy', '_BLOB.png')
    m101_p = p.replace('/ERSP_matrix/', '/ERSP_minus101/').replace('.npy', '_M101.png')
    return Path(blob_p), Path(m101_p)

if not GEN_ALT_VIEWS:
    print('[skip] GEN_ALT_VIEWS = False')
else:
    n_total = len(ersp_list)
    n_blob_wrote = n_blob_skip = 0
    n_m101_wrote = n_m101_skip = 0
    n_bad = 0
    print(f'Writing per-sample BLOB + M101 PNGs for {n_total} canonical samples...')
    for i in range(n_total):
        fp = df_meta.iloc[i].get('file_path')
        if not fp:
            n_bad += 1; continue
        blob_path, m101_path = _alt_view_paths(fp)
        if blob_path is None:
            n_bad += 1; continue
        try:
            if FORCE_REGEN or not blob_path.exists():
                save_sample_blob_png(ersp_list[i], blobs_per_sample[i], blob_path)
                n_blob_wrote += 1
            else:
                n_blob_skip += 1
            if FORCE_REGEN or not m101_path.exists():
                save_sample_minus101_png(ersp_list[i], blobs_per_sample[i], m101_path)
                n_m101_wrote += 1
            else:
                n_m101_skip += 1
        except Exception as e:
            print(f'  [warn] sample {i}: {e}')
            n_bad += 1
        if (i + 1) % 200 == 0:
            print(f'  ...{i+1}/{n_total}')
    print(f'BLOB PNGs : wrote {n_blob_wrote}, skipped existing {n_blob_skip}')
    print(f'M101 PNGs : wrote {n_m101_wrote}, skipped existing {n_m101_skip}')
    if n_bad:
        print(f'  ({n_bad} samples without parseable file_path — skipped)')


## Feature weighting

Apply per-feature-type weights to make blob-feature distance respect their relative importance.


In [ ]:
s23_init_blob_metric(
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    features_per_blob=8,
    feature_type_weights=FEATURE_TYPE_WEIGHTS,
)
from functions.lf_blob_metrics import BLOB_WEIGHT_VEC
Xw = X_blob * BLOB_WEIGHT_VEC
print('Xw:', Xw.shape)


# Clustering

Two methods on the same `Xw` (weighted blob features) on the canonical sample set.


In [ ]:
manifest_km = R.fit_and_save(
    Xw,
    df_keep=df_meta,
    method='kmeans',
    feature_set='blob',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means',
    feature_set_label='Blob Features',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/blob, by silhouette): {BEST_K}')


In [ ]:
manifest_hc = R.fit_and_save(
    Xw,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='blob',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    method_label='Hierarchical',
    feature_set_label='Blob Features',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/blob, by silhouette): {manifest_hc["summary"]["best_k"]}')


## Blob artifacts (for 231 minus101 to optionally reuse)

231 can rebuild blobs from scratch — but if it wants to skip s22 it can load these.


In [ ]:
import joblib

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)

def _save_blob_artifacts(manifest):
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    art_dir = run_dir / 'blob_artifacts'
    art_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(blobs_per_sample, art_dir / 'blobs_per_sample.joblib')
    with open(art_dir / 'segmentation_config.json', 'w') as f:
        import json as _json
        _json.dump({
            'valley_params': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                              for k, v in VALLEY_PARAMS.items()},
            'n_samples': int(len(blobs_per_sample)),
        }, f, indent=2, default=str)
    print(f'  blob_artifacts -> {art_dir}')

_save_blob_artifacts(manifest_km)
_save_blob_artifacts(manifest_hc)


## Per-cluster centroid PNGs (for the MOBA cluster chips)

For each cluster, finds the medoid (sample closest to cluster mean in `Xw` space)
and renders ITS ERSP + blobs in q34-style overlay. Single real sample per chip.


In [ ]:
import json
INDEX_PATH = CLUSTERING_DIR / 'index.json'
print('X_3d:', X_3d.shape, '  blobs_per_sample:', len(blobs_per_sample))

def _save_per_cluster_centroid_pngs_blob(manifest, ersp_3d_local, blobs_local):
    if manifest['feature_set'] != 'blob':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    df = pd.read_csv(run_dir / 'labels.csv')
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != ersp_3d_local.shape[0] or len(blobs_local) != len(labels):
        print(f"  [skip] {manifest['run_id']}: length mismatch")
        return 0
    X_train_path = run_dir / 'X_train.npy'
    if not X_train_path.exists():
        print(f"  [skip] {manifest['run_id']}: X_train.npy missing"); return 0
    X_train = np.load(X_train_path)
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        if len(idx) == 0: continue
        cluster_X = X_train[idx]
        centroid_xw = cluster_X.mean(axis=0)
        medoid_local = int(np.argmin(np.linalg.norm(cluster_X - centroid_xw, axis=1)))
        medoid_global = int(idx[medoid_local])
        fig, ax = plt.subplots(figsize=(2.4, 1.7))
        render_blob_overlay(ax, ersp_3d_local[medoid_global], blobs_local[medoid_global],
                            ersp_alpha=0.25, vmin=-5, vmax=5, linewidth=2.4, marker_size=56)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_dir / f'cluster_{int(c):02d}.png', dpi=90, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if INDEX_PATH.exists():
    with open(INDEX_PATH) as f:
        idx = json.load(f)
    runs = [r for r in idx.get('runs', []) if r['feature_set'] == 'blob']
    print(f'Backfilling for {len(runs)} blob runs...')
    for run in runs:
        mp = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not mp.exists(): continue
        manifest = json.loads(mp.read_text())
        n = _save_per_cluster_centroid_pngs_blob(manifest, X_3d, blobs_per_sample)
        if n: print(f"  [{manifest['method']}/blob] {manifest['run_id']} -> {n} PNGs")
    print('Done.')
